In [ ]:
"""
MicroLive Notebook
==================
This notebook requires MicroLive to be installed:
    pip install microlive

For development mode:
    pip install -e /path/to/microlive
"""
# MicroLive imports
from microlive import microscopy as mi
from microlive.utils.device import check_gpu_status

# Verify GPU support
check_gpu_status()

# Standard scientific imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# # Initial conditions
ki = 0.039  # Initiation rate
global_elongation_rate = 3.7  # Elongation rates for positions 1 to N-1
number_repetitions = 100
folding_delay = 0 #240
added_folding_delay = 0 
burnin_time = 1000
timePerturbationApplication = None #5*60
t_max = 360*5 #timePerturbationApplication + 25*60  # Maximum time
inhibitor_effectiveness=0.2
evaluatingInhibitor = 0


time_interval_between_frames_in_seconds = 1 # seconds
burnin = 500
downsample_time = 5
downsample_replicates = 1
percentage_to_remove_data = 0  # Remove 80% of the data
shift_data = True
simulate_photobleacing = False
correct_photobleaching = False
decay_rate = 0.9  # 1% decrease per minute


In [ ]:
# # # Initial conditions
# ki = 0.045  # Initiation rate
# global_elongation_rate = 7  # Elongation rates for positions 1 to N-1
# number_repetitions = 100
# folding_delay = 0 #240
# added_folding_delay = 0 
# burnin_time = 800
# timePerturbationApplication = None #5*60
# t_max = 360*5 #timePerturbationApplication + 25*60  # Maximum time
# inhibitor_effectiveness=0.2
# evaluatingInhibitor = 0


# time_interval_between_frames_in_seconds = 1 # seconds
# burnin = 500
# downsample_time = 5
# downsample_replicates = 1
# percentage_to_remove_data = 0  # Remove 80% of the data
# shift_data = True
# simulate_photobleacing = True
# correct_photobleaching = True
# decay_rate = 0.9  # 1% decrease per minute


In [ ]:
time_delay_to_codons = global_elongation_rate* folding_delay
time_delay_to_codons

In [ ]:
gfp_6_copies = True
if gfp_6_copies == True:
    file_path = pathlib.Path('pNZ212(pUB-mRuby2HA-6xsfGFP-24xMS2).dna')
else:
    file_path = pathlib.Path('pNZ251_pUB-mRuby2HA-1xsfGFP-24xMS2.dna')

file_path = pathlib.Path('/Users/nzlab-la/Desktop/micro/modeling/TASEP/pNZ208_pUB-24xUTagFullLength-KDM5B-MS2.dna')

# reading the sequence and extracting the elongation rates
protein, rna, dna, indexes_tags, _, seq_record, graphic_features  = read_sequence(seq=file_path, min_protein_length=50,TAG=TAGS)
plasmid_figure = plot_plasmid(seq_record, graphic_features,figure_width=25, figure_height=3)

gene_length = len(protein)+1 # adding 1 to account for the stop codon
tag_positions_first_probe_vector = indexes_tags[0]
tag_positions_second_probe_vector = indexes_tags[1] if len(indexes_tags) > 1 else None


if folding_delay > 0:
    # adding the folding delay as the time when the intensity is generated
    time_delay_to_codons = global_elongation_rate* folding_delay
    tag_positions_second_probe_vector = [tag_position + time_delay_to_codons for tag_position in tag_positions_second_probe_vector] if tag_positions_second_probe_vector is not None else None
    # remove the tags that are not in the gene
    tag_positions_second_probe_vector = [tag_position for tag_position in tag_positions_second_probe_vector if tag_position <= gene_length]
    if len(tag_positions_second_probe_vector) == 0:
        tag_positions_second_probe_vector = None
    print(tag_positions_second_probe_vector)

first_probe_position_vector = create_probe_vector(tag_positions_first_probe_vector, gene_length)
second_probe_position_vector = create_probe_vector(tag_positions_second_probe_vector, gene_length) if tag_positions_second_probe_vector is not None else None


In [ ]:
tag_positions_first_probe_vector[-1]

In [ ]:
file_path.name.split('.')[0]
plasmid_name = file_path.name.split('.')[0].replace('(','_').replace(')','_')
plasmid_name

In [ ]:
ke = calculate_codon_elongation_rates (rna, global_elongation_rate=global_elongation_rate)

In [ ]:
use_pause = False
if use_pause:
    # adding a pause site by setting the elongation rate to 0.001
    ke[-5] = 1/(global_elongation_rate+5) # 1/60 is the elongation rate of the pause site. Meaning this codon takes 60 seconds to be translated. 

## Deterministic modeling
____

In [ ]:
intensity_vector_first_signal_ode,intensity_vector_second_signal_ode = simulate_TASEP_ODE(ki, ke, gene_length, t_max,first_probe_position_vector,second_probe_position_vector,burnin_time)
# plt.plot(intensity_vector_first_signal_ode/np.max(intensity_vector_first_signal_ode))
# plt.plot(intensity_vector_second_signal_ode/np.max(intensity_vector_second_signal_ode))
# plt.show()


# Modeling TASEP SSA
____

In [ ]:
list_ribosome_trajectories, list_occupancy_output, matrix_intensity_first_signal_RT, matrix_intensity_second_signal_RT = simulate_TASEP_SSA(ki, ke, gene_length, t_max,number_repetitions, first_probe_position_vector,second_probe_position_vector,folding_delay=added_folding_delay,timePerturbationApplication=timePerturbationApplication, evaluatingInhibitor=evaluatingInhibitor,burnin_time=burnin_time,inhibitor_effectiveness=inhibitor_effectiveness)

In [ ]:
# calculate the mean and std of the matrix_intensity_first_signal_RT and matrix_intensity_second_signal_RT
mean_first_signal_RT = np.mean(matrix_intensity_first_signal_RT, axis=0)
sem_first_signal_RT = np.std(matrix_intensity_first_signal_RT, axis=0)/np.sqrt(number_repetitions)
if second_probe_position_vector is not None:
    mean_second_signal_RT = np.mean(matrix_intensity_second_signal_RT, axis=0)
    sem_second_signal_RT = np.std(matrix_intensity_second_signal_RT, axis=0)/np.sqrt(number_repetitions)

In [ ]:
# plot a single trajectory for the the two signals
plt.figure()
selected_trajectory = 0
plt.plot(matrix_intensity_first_signal_RT[selected_trajectory,:]/np.max(matrix_intensity_first_signal_RT[selected_trajectory,:]), label='first signal')
if second_probe_position_vector is not None:
    plt.plot(matrix_intensity_second_signal_RT[selected_trajectory,:]/np.max(matrix_intensity_second_signal_RT[selected_trajectory,:]), label='second signal')
plt.legend()
plt.show()

In [ ]:
# plot the mean and std as error shade
downsample = 50
downsampled_time = np.arange(0,t_max,downsample)

plt.figure(figsize=(5,4))
plt.plot(mean_first_signal_RT,color = 'k',linewidth=2, label='SSA')
plt.fill_between(np.arange(len(mean_first_signal_RT)), mean_first_signal_RT-sem_first_signal_RT, mean_first_signal_RT+sem_first_signal_RT, color='k', alpha=0.2)
plt.plot(downsampled_time, intensity_vector_first_signal_ode[::downsample], color = 'blue', linestyle='dashed', marker='o', label='ODE')
if second_probe_position_vector is not None:
    # plot for the second signal
    plt.plot(mean_second_signal_RT,color = 'k',linewidth=2, label='SSA')
    plt.fill_between(np.arange(len(mean_second_signal_RT)), mean_second_signal_RT-sem_second_signal_RT, mean_second_signal_RT+sem_second_signal_RT, color='k', alpha=0.2)
    plt.plot(downsampled_time, intensity_vector_second_signal_ode[::downsample], color = 'red', linestyle='dashed', marker='o', label='ODE')

plt.xlabel('Time')
plt.ylabel('Intensity')
plt.legend()
plt.show()

# Plot ribosome movement
___

In [ ]:
selected_trajectory = 0

#list_ribosome_trajectories, list_occupancy_output, matrix_intensity_first_signa_RT, matrix_intensity_second_signa_RT 
ribosome_trajectories = list_ribosome_trajectories[selected_trajectory]    
ribosome_trajectories = ribosome_trajectories[:,:]
intensity_vector_first_signal = matrix_intensity_first_signal_RT[selected_trajectory,:]
if second_probe_position_vector is not None:
    intensity_vector_second_signal = matrix_intensity_second_signal_RT[selected_trajectory,:]
else:
    intensity_vector_second_signal = None
#plot_RibosomeMovement(ribosome_trajectories, intensity_vector_first_signal ,tag_positions_first_probe_vector,SecondIntensityVector=intensity_vector_second_signal,second_probePositions=tag_positions_second_probe_vector,timePerturbationApplication=timePerturbationApplication) # intensity_vector_second_signal

In [ ]:
str_ki = str(ki).replace('.','_')
str_k = str(global_elongation_rate).replace('.','_')
fileNameGif = 'simulation_'+plasmid_name+'_ke_'+str_k+'_ki_'+str_ki + '_inhibitor_effectiveness_'+str(inhibitor_effectiveness)
plot_RibosomeMovement_and_Microscope(ribosome_trajectories, intensity_vector_first_signal, tag_positions_first_probe_vector, SecondIntensityVector=intensity_vector_second_signal, second_probePositions=tag_positions_second_probe_vector,FrameVelocity=20,timePerturbationApplication=timePerturbationApplication,fileNameGif=fileNameGif)

In [ ]:
# convert the frames to a gif
#import imageio
#imageio.mimsave('diffusion_simulation.gif', frames, duration=0.1)


## Calculating Correlations
____

In [ ]:
importlib.reload(mi)


matrix_intensity_first_signal_RT_downsampled = matrix_intensity_first_signal_RT[:,burnin:][::downsample_replicates,::downsample_time]
if second_probe_position_vector is not None:
    matrix_intensity_second_signal_RT_downsampled = matrix_intensity_second_signal_RT[:,burnin:][::downsample_replicates,::downsample_time]
    print('number of replicates : ', matrix_intensity_first_signal_RT_downsampled.shape[0], '\nnumber of time points : ', matrix_intensity_first_signal_RT_downsampled.shape[1])
    mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)
else:
    mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled)



In [ ]:
if simulate_photobleacing:
    decay_rate_first_signal = -np.log(decay_rate) / (100/downsample_time)  # 20% decrease after 100 minutes
    decay_rate_second_signal = -np.log(decay_rate) / (100/downsample_time) # 10% decrease after 100 minutes
    if second_probe_position_vector is not None:
        matrix_intensity_second_signal_RT_downsampled = simulate_photobleaching_in_trajectories(matrix_intensity_second_signal_RT_downsampled, decay_rate_second_signal)
        mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)
    else:
        matrix_intensity_first_signal_RT_downsampled = simulate_photobleaching_in_trajectories(matrix_intensity_first_signal_RT_downsampled, decay_rate_first_signal)
        mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled)


In [ ]:
if correct_photobleaching:
    decay_rate_first_signal = -np.log(decay_rate) / (100/downsample_time)  # 20% decrease after 100 minutes
    decay_rate_second_signal = -np.log(decay_rate) / (100/downsample_time) # 10% decrease after 100 minutes
    matrix_intensity_first_signal_RT_downsampled = correct_photobleaching_in_trajectories(matrix_intensity_first_signal_RT_downsampled, decay_rate_first_signal)
    if second_probe_position_vector is not None:
        matrix_intensity_second_signal_RT_downsampled = correct_photobleaching_in_trajectories(matrix_intensity_second_signal_RT_downsampled, decay_rate_second_signal)
        mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)
    else:
        mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled)


In [ ]:
# simulating misssing data.
if second_probe_position_vector is None:
    matrix_intensity_first_signal_RT_downsampled,matrix_intensity_second_signal_RT_downsampled = simulate_missing_data(matrix_intensity_first_signal_RT_downsampled, None, percentage_to_remove_data,replace_with='nan')
    mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled)
else:
    matrix_intensity_first_signal_RT_downsampled,matrix_intensity_second_signal_RT_downsampled = simulate_missing_data(matrix_intensity_first_signal_RT_downsampled,matrix_intensity_second_signal_RT_downsampled, percentage_to_remove_data,replace_with='nan')
    mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)


In [ ]:
# shift the data to the left
importlib.reload(mi)

if shift_data == True:
    matrix_intensity_first_signal_RT_downsampled  = mi.Utilities().shift_trajectories(matrix_intensity_first_signal_RT_downsampled, )
    mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled)
    if second_probe_position_vector is not None:
        matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled = mi.Utilities().shift_trajectories(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)
        mi.Plots().plot_matrix_sample_time(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)


In [ ]:
importlib.reload(mi)
mean_correlation_ch0, std_correlation_ch0, lags_ch0, correlations_array_ch0, dwell_time_ch0 = mi.Correlation(primary_data=matrix_intensity_first_signal_RT_downsampled, max_lag=None, nan_handling='ignore',shift_data=True,return_full=False,time_interval_between_frames_in_seconds=time_interval_between_frames_in_seconds*downsample_time,show_plot=True,start_lag=0,fit_type='linear',de_correlation_threshold=0.05).run()

In [ ]:
mi.Plots().plot_autocorrelation( correlations_array_ch0, mean_correlation_ch0,std_correlation_ch0,lags_ch0,5, plot_name='temp_AC.png', save_plots=True,)

In [ ]:
if second_probe_position_vector is not None:
    mean_correlation_ch1, std_correlation_ch1, lags_ch1, correlations_array_ch1, dwell_time_ch1 = mi.Correlation(primary_data=matrix_intensity_second_signal_RT_downsampled, max_lag=None, nan_handling='ignore',shift_data=True,return_full=False,time_interval_between_frames_in_seconds=time_interval_between_frames_in_seconds*downsample_time,show_plot=True,start_lag=0,fit_type='linear',de_correlation_threshold=0.001).run()

In [ ]:
importlib.reload(mi)
if second_probe_position_vector is not None:
    mean_cross_correlation, std_cross_correlation, lags_cross_correlation, cross_correlations_array, delay_cross_correlation = mi.Correlation(primary_data=matrix_intensity_first_signal_RT_downsampled, secondary_data=matrix_intensity_second_signal_RT_downsampled, max_lag=None, nan_handling='ignore', shift_data=True, return_full=True,time_interval_between_frames_in_seconds=time_interval_between_frames_in_seconds*downsample_time,show_plot=True).run()

In [ ]:
# plt.plot(lags_cross_correlation,mean_cross_correlation)
# # set the xlim between 1000 and 1500
# plt.xlim(-500,500)

# # plot a vertical line at zero
# plt.axvline(x=0, color='k', linestyle='--')
# # plot horizontal lines at 0.01
# plt.axhline(y=0.01, color='r', linestyle='--')
# plt.axhline(y=0.1, color='k', linestyle='--')
# plt.axvline(x=-folding_delay, color='k', linestyle='--')

# plt.show()

if second_probe_position_vector is not None:
    # calculate the first derivative of the cross correlation
    first_derivative = np.diff(mean_cross_correlation)
    # smooth the first derivative
    #first_derivative = np.convolve(first_derivative, np.ones(20)/20, mode='same')
    # plot the first derivative
    plt.figure(figsize=(5,3))
    plt.plot(lags_cross_correlation[:-1],first_derivative, label='first derivative', color='r', linewidth=4)
    plt.xlim(-500,500)

    plt.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    plt.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
    # add y label as the derivative of the cross correlation with symbols
    plt.ylabel('dG/dt')
    # x label is tau
    plt.xlabel(r'$\tau$'+ ' (au)')

    #plt.axvline(x=folding_delay, color='k', linestyle='--', linewidth=0.5)
    plt.axvline(x=-folding_delay, color='k', linestyle='--', linewidth=1)

    plt.show()



In [ ]:
print ('------------------------------------')
print('Decorrelation times: ')
estimated_decorrelation_time = np.round(  (gene_length-(np.max(tag_positions_first_probe_vector)//2) ) /global_elongation_rate   , 2)
print('Estimated decorrelaiton time', estimated_decorrelation_time )


#ke_calculated_ch0 =  np.round( (gene_length-np.max(tag_positions_first_probe_vector)/2) /dwell_time_ch0  , 2)
ke_calculated_ch0 =  np.round( gene_length /dwell_time_ch0  , 2)
ke_calculated_ch0_corrected =  np.round( (gene_length-(np.max(tag_positions_first_probe_vector))) /dwell_time_ch0  , 2)

print ('------------------------------------')

print('Elongation rates: ')
print('Calculated ch0', ke_calculated_ch0 , ' Corrected: ',ke_calculated_ch0_corrected, ', True: ',global_elongation_rate)

if second_probe_position_vector is not None:
    ke_calculated_ch1 = np.round( ( gene_length /dwell_time_ch1)  , 2)
    if second_probe_position_vector is not None:
        ke_calculated_ch1_corrected =  np.round( (gene_length-(np.max(tag_positions_second_probe_vector))) /dwell_time_ch1  , 2)
        print('Calculated ch1', ke_calculated_ch1, ', True: ',global_elongation_rate)

print ('------------------------------------')

print('Initiation rates: ')

# initiation rate
ki_calculated = np.round( 1/ (mean_correlation_ch0[0] * dwell_time_ch0), 3)
print('Calculated', ki_calculated, ', True: ',ki)

print ('------------------------------------')

print('Ribosomal occupancy: ')

# make a binary matrix of the ribosome trajectories that are more than 0
binary_matrix_ribosomal_occupancy = ribosome_trajectories > 0
binary_matrix_ribosomal_occupancy.shape
ribosomal_occupancy = np.sum(binary_matrix_ribosomal_occupancy, axis=0)
# calculate the ribosomal occupancy
theoretical_occupancy = np.round( (gene_length/global_elongation_rate) *ki , 2)
print( 'Calculated: ',np.round( np.mean(ribosomal_occupancy) ,2) , ', Theoretical: ',theoretical_occupancy)
print ('------------------------------------')

if second_probe_position_vector is not None:
    print('Folding delay: ')
    print('Calculated:' , delay_cross_correlation, ', True: ', folding_delay)
print ('------------------------------------')


In [ ]:
def detect_minima(array, threshold_percentage):
    minima_indices = []
    for sample_idx, sample in enumerate(array):
        # smooth the signal with a moving average of 5 points
        sample = np.convolve(sample, np.ones(10)/10, mode='same')
        # Normalize the sample between 0 and 100
        sample_normalized = (sample - np.min(sample)) / (np.max(sample) - np.min(sample)) * 100
        # Define the threshold value
        threshold = (threshold_percentage / 100.0) * np.max(sample_normalized)
        # Invert the signal to detect minima as peaks
        inverted_sample = -sample_normalized
        # Use find_peaks to detect peaks in the inverted signal
        peaks, properties = find_peaks(inverted_sample,height=-threshold )
        # Filter peaks based on the threshold
        valid_minima = peaks
        minima_indices.append(valid_minima)
    return minima_indices

def extract_windows(array, indices_list, window_size):
    windows = []
    for sample_idx, indices in enumerate(indices_list):
        sample_windows = []
        for idx in indices:
            start = idx - window_size
            end = idx + window_size + 1
            if start >= 0 and end <= array.shape[1]:
                window = array[sample_idx, start:end]
                sample_windows.append(window)
        if sample_windows:
            windows.append(np.array(sample_windows))
        else:
            windows.append(np.array([]))
    return windows

def calculate_average_profiles(windows):
    avg_profiles = []
    for sample_windows in windows:
        if sample_windows.size == 0:
            avg_profiles.append(None)
            continue
        # Average across all windows at each time point
        avg_profile = np.mean(sample_windows, axis=0)
        avg_profiles.append(avg_profile)
    return avg_profiles

def generate_control_indices(array_shape, n_control_points, window_size):
    control_indices = []
    num_samples = array_shape[0]
    num_timepoints = array_shape[1]
    min_index = window_size
    max_index = num_timepoints - window_size - 1
    for sample_idx in range(num_samples):
        if max_index < min_index:
            control_indices.append(np.array([], dtype=int))
            continue
        possible_indices = np.arange(min_index, max_index + 1)
        n_sample = min(n_control_points, len(possible_indices))
        random_indices = np.random.choice(possible_indices, size=n_sample, replace=False)
        control_indices.append(random_indices)
    return control_indices

def plot_results(avg_profiles_1, avg_profiles_2, control_profiles_1, control_profiles_2, window_size):
    time_points = np.arange(-window_size, window_size + 1)
    # Collect all non-None profiles
    valid_avg_profiles_1 = []
    valid_avg_profiles_2 = []
    valid_control_profiles_1 = []
    valid_control_profiles_2 = []
    for i in range(len(avg_profiles_1)):
        if avg_profiles_1[i] is not None:
            valid_avg_profiles_1.append(avg_profiles_1[i])
        if avg_profiles_2[i] is not None:
            valid_avg_profiles_2.append(avg_profiles_2[i])
        if control_profiles_1[i] is not None:
            valid_control_profiles_1.append(control_profiles_1[i])
        if control_profiles_2[i] is not None:
            valid_control_profiles_2.append(control_profiles_2[i])
    # Check if there are any valid profiles
    if not valid_avg_profiles_1 or not valid_control_profiles_1:
        print("No valid profiles to plot.")
        return
    # Stack the profiles and compute the overall average
    overall_avg_profile_1 = np.mean(np.vstack(valid_avg_profiles_1), axis=0)
    overall_avg_profile_2 = np.mean(np.vstack(valid_avg_profiles_2), axis=0)
    overall_control_profile_1 = np.mean(np.vstack(valid_control_profiles_1), axis=0)
    overall_control_profile_2 = np.mean(np.vstack(valid_control_profiles_2), axis=0)
    
    # Normalize the profiles between 0 and 1
    def normalize_profile(profile):
        return (profile - np.min(profile)) / (np.max(profile) - np.min(profile))
    
    #overall_avg_profile_1 = normalize_profile(overall_avg_profile_1)
    #overall_avg_profile_2 = normalize_profile(overall_avg_profile_2)
    #overall_control_profile_1 = normalize_profile(overall_control_profile_1)
    #overall_control_profile_2 = normalize_profile(overall_control_profile_2)
    # Plot the overall average profiles
    plt.figure(figsize=(7, 5))
    #plt.plot(time_points, overall_avg_profile_1, label='Signal 1', color='blue', linewidth=4)
    plt.plot(time_points, overall_avg_profile_2, label='Signal 2', color='red', linewidth=4)
    #plt.plot(time_points, overall_control_profile_1, label='Control Signal 1', linestyle=':', color='blue', alpha=0.5)
    plt.plot(time_points, overall_control_profile_2, label='Control Signal 2', linestyle=':', color='red', alpha=0.5)
    # plot a vertical line at time 0
    plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
    plt.axvline(x=folding_delay, color='k', linestyle='--')
    # plot a verticla line when the overall_avg_profile_2 is minimum
    min_index = np.argmin(overall_avg_profile_2)
    plt.axvline(x=time_points[min_index], color='red', linestyle='--', linewidth=1)
    print('Intensity surronding minima, Delay: ', time_points[min_index])
    plt.title('Intensity Profiles')
    plt.xlabel('Time Relative to Minima')
    plt.ylabel('Intensity')
    # add legend outside the plot to the top right
    plt.legend(loc='upper right')
    #plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

def analyze_delay(array_1, array_2, threshold_percentage, window_size, n_control_points):
    # Detect minima in array_1
    minima_indices_1 = detect_minima(array_1, threshold_percentage)
    # Extract windows around minima in both arrays
    windows_1 = extract_windows(array_1, minima_indices_1, window_size)
    windows_2 = extract_windows(array_2, minima_indices_1, window_size)
    # Calculate average intensity profiles
    avg_profiles_1 = calculate_average_profiles(windows_1)
    avg_profiles_2 = calculate_average_profiles(windows_2)
    # Generate control data
    control_indices = generate_control_indices(array_1.shape, n_control_points, window_size)
    # Extract windows around control indices in both arrays
    control_windows_1 = extract_windows(array_1, control_indices, window_size)
    control_windows_2 = extract_windows(array_2, control_indices, window_size)
    # Calculate control average intensity profiles
    control_profiles_1 = calculate_average_profiles(control_windows_1)
    control_profiles_2 = calculate_average_profiles(control_windows_2)
    # Plot the results
    plot_results(avg_profiles_1, avg_profiles_2, control_profiles_1, control_profiles_2, window_size)


In [ ]:
# Parameters
threshold_percentage = 20  # User-defined threshold percentage
window_size = 300            # Number of values before and after the minima
n_control_points = 10      # Number of random positions per sample for control data

# Run the analysis
analyze_delay(matrix_intensity_first_signal_RT, matrix_intensity_second_signal_RT, threshold_percentage, window_size, n_control_points)
analyze_delay(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled, threshold_percentage, window_size, n_control_points)
